In [ ]:
'''monte carlo simulation'''


'''simulation of stock terminal price movements'''
S0 = 9.15 # stock price at time 0 当前价格
T = 1 # 1 year of obsevation 观察期
n_steps = 100 # number of steps 小时间步长离散化，把一年分成100个小时间段
mu = 0.15 # annual return mean 年收益率
sigma = 0.2 # annual volatility 年波动率

np.random.seed(123456)
n_simulations = 1000  #做1000次模拟

dt = T/n_steps  #时间离散，每一步的时间长度
S = np.zeros([n_simulations], dtype=float)
#做了1000次模拟，每次模拟都生成一个终端价格，存储在S数组中，初始化为0
x = np.arange(0, int(n_steps), 1) #用于循环便利100个小时间步

for j in np.arange(0, n_simulations):
    tt = S0
    for i in x:
        e = np.random.normal()
        tt += tt*(mu-0.5*pow(sigma, 2))*dt + tt*sigma*e*np.sqrt(dt)
    S[j] = tt
plt.xlabel("terminal price")
plt.ylabel("number of frequencies")
plt.hist(S)
plt.show()



'''call simulation'''
S0 = 40
X = 40
T = 0.5
r = 0.05
sigma = 0.2
n_steps = 100

np.random.seed(123456)
n_simulations = 5000
dt = T/n_steps
call = np.zeros([n_simulations], dtype=float)

x = np.arange(0, int(n_steps), 1)

for j in np.arange(0, n_simulations):
    sT = S0
    for i in x[:-1]:
        e = np.random.normal()
        a = sigma *e*np.sqrt(dt)
        sT = sT * exp((r - 0.5*sigma*sigma)*dt + a)
        call[j] = max(sT - X, 0)

call_price = exp(-r*T)*call.mean()
print(f"the price of the call option is {call_price}")


'''path of stock price change'''
S0 = 9.15

T = 1
n_steps = 100
mu = 0.15
sigma = 0.2

np.random.seed(123456)
n_simulation = 5

dt = T/n_steps
S = np.zeros([n_simulation], dtype=float)
x = np.arange(0, int(n_steps), 1)

for j in np.arange(0, n_simulation):
    S[0] = S0
    for i in x[:-1]:
        e = np.random.normal()
        aa = sigma * S[i]*sqrt(dt)*e
        S[i+1] = S[i] + S[i]*(mu - 0.5*sigma*sigma)*dt + aa
    plt.plot(x, S)

plt.xlabel("time step")
plt.ylabel("stock price")
plt.show()


'''correlated random series'''

np.random.seed(123456)
n = 1000
rho = 0.3
x1 = np.random.normal(size = n)
x2 = np.random.normal(size = n)

y1 = x1
y2 = rho * x1 + np.sqrt(1-rho**2) * x2

corMatrix = np.corrcoef(y1, y2)
print(corMatrix)


'''calculating mean and std then simulate'''
ticker = "WMT"
n_share = 500
confidence_level = 0.99
begdate = "2012-1-1"
enddate = "2016-12-31"
df = yf.download(ticker, begdate, enddate)
df = np.read_pickle("WMT.pkl")

ret = df["Close"].pct_change().dropna()
position = round(n_share * df["Close"].iloc[-1], 2)

std = ret.std()

n_simulations = 5000
np.random.seed(123456)
ret2 = np.random.normal(ret.mean(), std, n_simulations)
rer3 = np.sort(ret2)
m = int(n_simulations * (1 - confidence_level))

VaR_simulation = round(position * rer3[m], 2)
print(f"VaR by simulation is {VaR_simulation} tomorrow")


'''up and out call'''

def bsCall (S, X, T, r, sigma):
    d1 = (log(S/X) + (r + sigma*sigma/2)*T)/(sigma * sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S*stats.norm.cdf(d1) - X*exp(-r*T)*stats.norm.cdf(d2)

def up_and_out_call(S0, X, T, r, sigma, n_simulation, barrier):
    n_steps = 100
    dt = T/n_steps
    total = 0
    for i in np.arange(0, n_simulation):
        sT = S0
        knocked_out = False
        for j in np.arange(0, int(n_steps)):
            e = np.random.normal()
            a = sigma *e*np.sqrt(dt)
            sT = sT * exp((r - 0.5*sigma*sigma)*dt + a)
            if sT >= barrier:
                knocked_out = True
                break
            if knocked_out == False:
                total += bsCall(sT, X, T - j*dt, r, sigma)
    return total/n_simulation

S0 = 40
X = 40
T = 0.5
r = 0.05
sigma = 0.2
n_simulation = 1000
barrier = 42

call=round(bsCall(S0, X, T, r, sigma), 2 )
uac=round(up_and_out_call(S0, X, T, r, sigma, n_simulation, barrier), 2)
print(f"the price of the vanilla call option is {call}")
print(f"the price of the up and out call option is {uac}")



In [ ]:
#linking two methods for var using simulation
position=1e6
std=0.2
mean=0.08
confidence_level=0.99
n_simulation=50000
#method1
z=stats.norm.ppf(1-confidence_level)
VaR=round(position*(mean-z*std), 2)
print(f"the VaR by method 1 is {VaR}")
#method2 monte carlo simulation
np.random.seed(123456)
ret2=np.random.normal(mean, std, n_simulation)
ret3=np.sort(ret2)
m=int(n_simulation*(1-confidence_level))
VaR2=round(position*ret3[m], 2)
print(f"the VaR by method 2 is {VaR2}")

In [ ]:
#long-term return comparison
import yfinance as yf
ticker='ibm'
begdate='2012-1-1'
enddate='2016-12-31'
n_forecast=25
df=yf.download(ticker, begdate, enddate)
df=pd.read_pickle("ibm.pkl")
df[retplus1]=df["Close"].pct_change().dropna() + 1
df['year'] = df.index.year
df=df.dropna()
retannual=df.retplus1.groupby(df.year).prod() - 1
retannual.columns=['retannual']

n_history=retannual.count()
a_mean=round(retannual.mean(), 4)
retplus1=[x+1 for x in retannual]
g_mean=round(stats.gmean(retplus1)-1, 4)
w=round(n_forecast/n_history, 4)
future_return=round(w*a_mean + (1-w)*g_mean, 4)
print(f'arithmetic mean is {a_mean}')
print(f'geometric mean is {g_mean}')
print(f'future return is {future_return}')

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from math import exp, log, sqrt
from scipy import stats

np.random.seed(123456)
x = np.random.standard_normal(size = 10) #从标准正态分布（0，1）中生成10个随机数
print(x)

[ 0.4691123  -0.28286334 -1.5090585  -1.13563237  1.21211203 -0.17321465
  0.11920871 -1.04423597 -0.86184896 -2.10456922]


In [ ]:
'''randomly select 10 stocks among NYSE stocks'''
infile = 'http://datayyy.com/data_pickle/nyseList.pickle'
df = pd.read_pickle(infile)

nStock = 10

np.random.seed(123456)
a = np.random.choice(df.ticker, nStock)
#np.random.choice(a, size=None, replace=True, p=None)随机抽样函数，a是从中抽样的数组，size是抽样的大小，replace是是否有放回抽样，p是每个元素被抽中的概率
stock_chosen = df.loc[df.index[a]] #根据索引a从df中选取对应的行


In [ ]:
#进行投骰子模拟
'''roll a dice function'''
def roll_dice(n):
    return np.random.randint(1, 7, n)   #从1到6中生成n个随机整数

res = []
for i in np.arange(0, 10):
    res.append(roll_dice(10))   #进行10次投骰子模拟，每次模拟投10个骰子
print(res)

[array([3, 3, 3, 5, 6, 5, 5, 5, 5, 6]), array([4, 6, 2, 5, 4, 4, 6, 1, 5, 3]), array([3, 1, 4, 5, 3, 2, 3, 5, 5, 5]), array([5, 6, 5, 5, 3, 2, 3, 5, 2, 5]), array([3, 5, 6, 3, 6, 1, 4, 5, 5, 3]), array([2, 4, 3, 4, 2, 5, 1, 4, 1, 6]), array([2, 5, 5, 3, 3, 6, 6, 2, 1, 4]), array([1, 5, 5, 1, 5, 5, 1, 6, 4, 6]), array([2, 5, 5, 5, 3, 4, 2, 2, 3, 3]), array([3, 6, 5, 6, 4, 3, 6, 4, 5, 5])]


In [3]:
'''the permutation steps in randomization'''#洗牌，打乱
x = np.arange(1, 11)    #生成一个包含1到10的数组x
print(np.random.permutation(x)) #打乱x中的元素顺序，返回一个新的数组

for i in np.arange(1, 6):
    y = np.random.permutation(x)
    print(f"the {i}th permutation is {y}")

[ 8 10  6  1  7  9  4  3  5  2]
the 1th permutation is [ 8  4  7  6  9  3  5  2  1 10]
the 2th permutation is [ 5  3 10  9  8  6  4  1  2  7]
the 3th permutation is [ 6  5  9  7  3  4  2  8  1 10]
the 4th permutation is [10  6  2  1  5  7  8  3  9  4]
the 5th permutation is [ 8  9  6  4  3  2 10  1  5  7]
